In [ ]:
import spacy
import nltk
import pandas as pd
import matplotlib.pyplot as plt
from wordcloud import WordCloud
from google.colab import drive
from collections import Counter
from transformers import pipeline
import itertools
import re
import math

In [ ]:
nltk.download('wordnet')
nltk.download('stopwords')

In [ ]:
!python -m spacy download en_core_web_lg

In [ ]:
drive.mount('/content/drive')

In [ ]:
pd.set_option('display.max_colwidth', 100)

In [ ]:
lifestyle_words = ["relationship", "bills", "expenses", "cuisine", "food", "wellness", "fitness", "health", "nutrition", "exercise", "diet", "selfcare", "mindfulness", "hobbies", "leisure", "happiness", "routine", "habits", "productivity", "motivation", "personal growth", "minimalism", "organization", "time management", "family", "relationships", "travel", "adventure", "vacation", "home decor", "fashion", "beauty", "style", "clothes", "technology", "gadgets", "social media", "sleep", "dietary restrictions", "organic food", "vegan", "stress management", "personal finance", "budgeting", "money management",  "work life balance", "self-improvement", "spirituality", "non vegetarian" ]
employment_words = ["recruit", "startup", "corporate", "colleague", "entrepreneurship", "job","work","career","occupation","profession","hiring","recruitment","employee","workforce","salary","wage","internship","contract","full-time","part-time","freelance","unemployment","promotion","productivity","skillset","industry","job market","resignation","onboarding"]
safety_words = ["fraud", "scam", "threaten", "greivance", "laundering", "misbehaves", "customs", "drugs", "journalists", "evidence", "suspects", "abuse", "harass", "risks", "maniacs", "lawyer", "security", "protection", "crime", "law", "policing", "surveillance", "emergency", "accidents", "fire safety", "road safety", "community", "neighborhood", "vandalism", "violence", "self-defense", "public safety", "rescue", "patrol", "incident", "evacuation"]
healthcare_words = ["suicide", "depression", "lonely", "medicine", "hospitals", "clinics", "doctors", "nurses", "health", "treatment", "diagnosis", "care", "wellness", "emergency", "ambulance", "pharmacy", "health insurance", "checkups", "mental health", "surgery", "vaccination", "rehabilitation", "therapy", "primary care", "health services", "patient", "telemedicine" ]
education_words = ["iit", "iisc", "nit", "learning", "schools", "teachers", "students", "classroom", "university", "college", "curriculum", "education system", "academic", "knowledge", "literacy", "subjects", "exams", "scholarship", "tuition", "degree", "training", "skills", "e-learning", "study", "research", "lectures", "homework", "assignments", "grades", "online education", "education resources", "mooc" ]
transportation_words = [ "commute", "traffic", "vehicles", "buses", "trains", "metro", "roads", "highways", "cars", "bicycles", "motorcycles", "public transport", "transit", "logistics", "routes", "stations", "airports", "railways", "bridges", "ferries", "taxi", "ridesharing", "pedestrian", "walkways", "transport network", "mobility", "cargo", "freight", "delivery", "traffic congestion", "transportation services"]
utilities_words = [ "electricity", "water", "gas", "sewage", "waste management", "sanitation", "energy", "power supply", "renewable energy", "solar power", "wind energy", "internet", "broadband", "telecommunication", "mobile network", "fuel", "piped gas", "grid", "water supply", "plumbing", "recycling", "waste disposal", "utility bills", "smart meters", "maintenance"]
other_infrastructure_words = ["seaports", "infrastructure", "telecommunication", "buildings", "hospitals", "schools", "parks", "public utilities", "drainage",  "irrigation", "urban planning", "construction", "housing", "infrastructure development", "smart cities", "pipelines", "waste management", "infrastructure projects", "industrial zones", "sustainable infrastructure", "transport hubs" ]
quality_words = [ "pollution", "contamination", "air quality", "clean air", "chemicals", "emissions", "smoking", "carbon dioxide", "greenhouse gases", "smog", "airborne particles", "water quality", "clean water", "freshwater", "marine pollution", "ocean health", "groundwater", "water bodies", "sanitation", "soil quality", "land pollution", "erosion", "pesticides", "deforestation", "fertility", "reforestation", "biodiversity", "sustainability", "ecosystem health", "wastewater", "recycling", "microplastics", "hazardous waste", "organic farming", "renewable resources", "air monitoring", "climate change", "environmental protection" ]
other_environment_words = ["rainfall", 'biodegradable',  'carbon footprint',  'climate',  'conservation',  'ecology',  'ecosystem',  'environmental',  'forest',  'global warming',  'green energy',  'habitat',  'natural disaster',  'natural resources',  'nature reserves',  'organic',  'renewable energy',  'soil erosion',  'water conservation',  'wildlife',  'wildlife protection']
politics_words = [ "democracy", "elections", "vote", "politician", "policy", "political party", "congress", "senate", "parliament", "campaign", "voter", "legislation", "judiciary", "laws", "constitution", "rights", "freedom", "equality", "justice", "opposition", "social justice", "activism", "reform", "nationalism", "socialism", "capitalism", "libertarian", "civic duty", "taxes",  "foreign policy", "diplomacy", "international relations", "treaty", "embassy", "sanctions", "war", "peace talks", "national security", "propaganda", "political ideology", "referendum", "impeachment", "coup", "autocracy"]
other_government_words = ['administration',  'authorities',  'bureaucracy',  'cabinet',  'central government',  'civil rights',  'civil service',  'constitutional monarchy',  'decentralization',  'federal',  'governance',  'government agency',  'governor',  'legislative body',  'local government',  'mayor',  'minister',  'national government',  'policy maker',  'policy reform',  'public funding',  'public interest',  'public policy',  'public sector',  'public servant',  'regulations',  'sovereignty',  'state',  'subsidies']

In [ ]:
categ_list = ["quality of life", "infrastructure", "environment", "governance"]
category_dict = {"quality of life": ["lifestyle", "employment","safety","healthcare","education"],"infrastructure":["transportation","utilities"],"environment": ["air, water, land quality"],"governance": ["politics", "government"] }
sub_categ = ["lifestyle", "employment", "safety","healthcare","education", "transportation","utilities", "air, land, water quality","politics", "government"]
similarity_check_order = [lifestyle_words, employment_words, safety_words, healthcare_words, education_words, transportation_words, utilities_words, other_infrastructure_words, quality_words, other_environment_words, politics_words, other_government_words]
categ_reverse = ["quality of life", "quality of life", "quality of life", "quality of life", "quality of life", "infrastructure", "infrastructure", "infrastructure", "environment", "environment", "governance", "governance"]
sub_categ_reverse = ["lifestyle", "employment", "safety","healthcare","education", "transportation","utilities", "other infrastructure","air, land, water quality", "other environment","politics", "government"]

In [ ]:
def get_score(nlp, word_tokens, target_word_token):
  score = [k.similarity(target_word_token) for k in word_tokens]
  return sum(sorted(score, reverse=True)[0:1])/1

def get_similarity(topics, categories, threshold=0.55):
  """Classify topics based on similarity with given categories."""
  nlp = spacy.load('en_core_web_lg')
  topic_dict = {}
  unachieved_topics = []
  category_token_list = [None for category in categories]
  for i in range(len(categories)):
    temp_list = [nlp(word) for word in categories[i]]
    category_token_list[i] = temp_list
  for topic in topics:
    topic_token = nlp(topic)
    all_scores = [get_score(nlp, category_token_list[i], topic_token) for i in range(len(category_token_list))]
    max_score = max(all_scores)
    best_score_index = all_scores.index(max_score)
    if max_score >= threshold:
      if topic not in topic_dict:
        topic_dict[topic] = []
      topic_dict[topic].append(categ_reverse[best_score_index])
      topic_dict[topic].append(sub_categ_reverse[best_score_index])
    else:
      unachieved_topics.append(topic)
  return topic_dict, unachieved_topics


In [ ]:
def merge_aspects(sent_dict, aspect_dict, categ_list):
  categ_count = {i:0 for i in categ_list}
  for sent in sent_dict:
    if sent_dict[sent] is not None:
      if sent in aspect_dict:
        categ_count[aspect_dict[sent][0]] += 1
  max_categ = max(categ_count, key=categ_count.get)
  if categ_count[max_categ] > 0:
    return max_categ
  else:
    return "no applicable category"

def sent_to_score(sent):
  sentiment = sent.lower()
  score = 0
  if sentiment == "positive":
    score = 1
  elif sentiment == "neutral":
    score = 0
  elif sentiment == "negative":
    score = -1
  return score

def fix_aspects(sent_dict, aspect_dict, sub_categ_list):
  sub_categ_count = {}
  for sent in sent_dict:
    if sent_dict[sent] is not None:
      if sent in aspect_dict:
        if aspect_dict[sent][1] not in sub_categ_count:
          sub_categ_count[aspect_dict[sent][1]] = {"count":0,"score":0}
        sub_categ_count[aspect_dict[sent][1]]["count"] += 1
        sub_categ_count[aspect_dict[sent][1]]["score"] += sent_to_score(sent_dict[sent])
  if len(sub_categ_count) == 0:
    sub_categ_count["no applicable subcategory"] = {"count":1,"score":0}
  return sub_categ_count

def delete_unnecessary_aspects(sent_dict):
  new_dict = {}
  for sent in sent_dict:
    if sent_dict[sent] is not None:
      new_dict[sent] = sent_dict[sent]
  return new_dict

In [ ]:
def score_to_sent(score):
  if score > 0:
    return "positive"
  elif score == 0:
    return "neutral"
  else:
    return "negative"

In [ ]:
def overall_post_sentiment_using_aspect(df_inp):
  df = df_inp.copy()
  sentiments = []
  sign = lambda x: math.copysign(1 if x!=0 else 0, x)
  for aspects in df['subcategory']:
    temp_sent_score = 0
    for aspect, sentiment_data in aspects.items():
      temp_sent_score += sign(sentiment_data['score'])
    sentiments.append(score_to_sent(temp_sent_score))
  df['overall_sent_aspect'] = sentiments
  return df

In [ ]:
def generate_wordcloud(df):
  all_aspects = []
  for aspects in df['subcategory']:
    all_aspects.extend(aspects.keys())
  wordcloud = WordCloud(width=800, height=400, background_color='white').generate(' '.join(all_aspects))
  plt.figure(figsize=(10, 5))
  plt.imshow(wordcloud, interpolation='bilinear')
  plt.axis('off')
  plt.title("Frequently Discussed Aspects")
  plt.show()

In [ ]:
def sentiment_distribution_and_top_terms(df, order = None):
  """Display sentiment distribution as a bar chart"""
  sentiment_distribution = {}
  for aspects in df['subcategory']:
    for aspect, sentiment_data in aspects.items():
      sentiment = score_to_sent(sentiment_data['score'])
      if aspect not in sentiment_distribution:
        sentiment_distribution[aspect] = {"positive": 0, "neutral": 0, "negative": 0}
      sentiment_distribution[aspect][sentiment] += 1
  if order is not None:
    aspect_names = order
  else:
    aspect_names = list(sentiment_distribution.keys())
  positive_counts = [sentiment_distribution[aspect]["positive"] for aspect in aspect_names]
  neutral_counts = [sentiment_distribution[aspect]["neutral"] for aspect in aspect_names]
  negative_counts = [sentiment_distribution[aspect]["negative"] for aspect in aspect_names]
  bar_width = 0.3
  x = range(len(aspect_names))
  plt.figure(figsize=(12, 6))
  plt.bar(x, positive_counts, width=bar_width, label='Positive', color='green', alpha=0.7)
  plt.bar([p + bar_width for p in x], neutral_counts, width=bar_width, label='Neutral', color='blue', alpha=0.7)
  plt.bar([p + 2*bar_width for p in x], negative_counts, width=bar_width, label='Negative', color='red', alpha=0.7)
  plt.xlabel("Aspects")
  plt.ylabel("Number of Posts")
  plt.title("Sentiment Distribution by Aspect")
  plt.xticks([p + bar_width for p in x], aspect_names, rotation=90)
  plt.legend()
  plt.show()
  print("Sentiment Distribution by Aspect:")
  for aspect, sentiment_counts in sentiment_distribution.items():
    print(f"{aspect}: {sentiment_counts}")


In [ ]:
def aspect_importance(df):
  aspect_importance = Counter()
  for idx, row in df.iterrows():
    for aspect, details in row['subcategory'].items():
      importance = row['total_votes']
      aspect_importance[aspect] += importance
  print("Aspect Importance (based on User Engagement):")
  for aspect, importance in aspect_importance.most_common():
    print(f"{aspect}: {importance}")

def aspect_importance_score(df):
  aspect_importance = Counter()
  for idx, row in df.iterrows():
    for aspect, details in row['subcategory'].items():
      importance = row['score']
      aspect_importance[aspect] += importance
  print("Aspect Importance (based on score):")
  for aspect, importance in aspect_importance.most_common():
    print(f"{aspect}: {importance}")

def aspect_importance_upvotes(df):
  aspect_importance = Counter()
  for idx, row in df.iterrows():
    for aspect, details in row['subcategory'].items():
      importance = row['upvotes']
      aspect_importance[aspect] += importance
  print("Aspect Importance (based on upvotes):")
  for aspect, importance in aspect_importance.most_common():
    print(f"{aspect}: {importance}")

In [ ]:
def preprocess_text(text):
  """Remove non-alphabetic characters, convert to lowercase, and remove stop words."""
  stop_words = set(nltk.corpus.stopwords.words('english'))
  text = re.sub(r'[^a-zA-Z ]', '', text)  # Remove non-alphabetic characters
  tokens = text.lower().split()  # Tokenize and convert to lowercase
  tokens = [word for word in tokens if word not in stop_words]  # Remove stop words
  return tokens

def get_word_frequencies(df, aspect_column, text_column, aspect_name):
  relevant_texts = []
  for index, row in df.iterrows():
    aspects = row[aspect_column]
    if aspect_name in aspects:
      relevant_texts.append(row[text_column])
  all_tokens = []
  for text in relevant_texts:
    all_tokens.extend(preprocess_text(text))
  word_freq = Counter(all_tokens)
  return word_freq

def plot_barchart(word_freq, aspect_name, top_n=10):
  top_words = word_freq.most_common(top_n)
  words, counts = zip(*top_words)
  plt.figure(figsize=(8, 6))
  plt.bar(words, counts, color='skyblue')
  plt.title(f"Top {top_n} Words for Aspect: {aspect_name}")
  plt.xlabel("Words")
  plt.ylabel("Frequency")
  plt.xticks(rotation=45)
  plt.show()

def plot_wordcloud(word_freq, aspect_name):
  wordcloud = WordCloud(width=800,height=400,background_color='white').generate_from_frequencies(word_freq)
  plt.figure(figsize=(10, 6))
  plt.imshow(wordcloud, interpolation="bilinear")
  plt.title(f"Word Cloud for Aspect: {aspect_name}")
  plt.axis('off')
  plt.show()

def visualize_aspects(df, aspect_column, text_column, aspects, top_n=15):
  """Visualize aspects using word cloud and bar plot on whole text """
  for aspect in aspects:
    print(f"Visualizing Aspect: {aspect}")
    word_freq = get_word_frequencies(df, aspect_column, text_column, aspect)
    plot_barchart(word_freq, aspect, top_n)
    plot_wordcloud(word_freq, aspect)


In [ ]:
def transform_dataframe(df, aspect_column, topics_column):
  """Helper function to flatten dictionary columns into aspect, terms"""
  rows = []
  for index, row in df.iterrows():
    topics = row[topics_column]
    aspects = row[aspect_column]
    for topic, topic_sentiment in topics.items():
      for aspect, aspect_sentiment in aspects.items():
        rows.append({"topics": topic, "aspects": aspect, "aspect_sentiment": score_to_sent(aspect_sentiment['score'])})
  return pd.DataFrame(rows)

def topic_similarity_to_df(topic_similarity):
  columns = {"topics": [], "subcategory":[], "category":[]}
  for topic, category_list in topic_similarity.items():
    columns["topics"].append(topic)
    columns["category"].append(category_list[0])
    columns["subcategory"].append(category_list[1])
  df = pd.DataFrame(columns)
  return df

def get_top_terms_per_aspect(dataframe, sentiment, aspect_column, term_column, aspect_sentiment_column, top_n=15):
  """Function to get top N terms per aspect for a given sentiment"""
  aspect_terms = {}
  filtered_df = dataframe[dataframe[aspect_sentiment_column] == sentiment]
  for aspect in filtered_df[aspect_column].unique():
    aspect_df = filtered_df[filtered_df[aspect_column] == aspect]
    term_counts = Counter(aspect_df[term_column])
    aspect_terms[aspect] = term_counts.most_common(top_n)
  return aspect_terms

def plot_barchart_per_aspect(aspect_terms, sentiment):
  """Plot bar chart for given top N terms per aspect for a given sentiment"""
  for aspect, terms in aspect_terms.items():
    if terms:
      terms, counts = zip(*terms)
      plt.figure(figsize=(8, 5))
      plt.bar(terms, counts, color="skyblue")
      plt.title(f"Top Terms for Aspect '{aspect.capitalize()}' ({sentiment.capitalize()})")
      plt.xlabel("Terms")
      plt.ylabel("Frequency")
      plt.xticks(rotation=45)
      plt.show()




def plot_wordclouds_by_aspect(df, sentiments, aspect_column, topic_column, aspect_sentiment_column):
  """Plot word clouds for each aspect and sentiment using only topic terms"""
  for aspect in df[aspect_column].unique():
    valid_sentiments = []
    valid_wordclouds = []

    for i, sentiment in enumerate(sentiments):
      sentiment_text = " ".join(df[(df[aspect_column] == aspect) & (df[aspect_sentiment_column] == sentiment)][topic_column]) # Filter topics for the current aspect and sentiment
      if sentiment_text.strip():  # Check if there are terms for this sentiment
        valid_sentiments.append(sentiment)
        wordcloud = WordCloud(width=400, height=400, background_color='white').generate(sentiment_text)
        valid_wordclouds.append(wordcloud)

    if not valid_sentiments:
      print(f"No terms found for aspect: {aspect}")
      continue

    fig, axs = plt.subplots(1, len(valid_sentiments), figsize=(6 * len(valid_sentiments), 6))
    if len(valid_sentiments) == 1:
      axs = [axs]
    for i, (sentiment, wordcloud) in enumerate(zip(valid_sentiments, valid_wordclouds)):
      axs[i].imshow(wordcloud, interpolation='bilinear')
      axs[i].set_title(f"{sentiment.capitalize()} Word Cloud - Aspect: {aspect}")
      axs[i].axis('off')
    plt.tight_layout()
    plt.show()


In [ ]:
def plot_barplots_by_aspect(df, sentiments, aspect_column, topic_column, aspect_sentiment_column):
  for aspect in df[aspect_column].unique():
    valid_sentiments = []
    valid_counts = []
    for sentiment in sentiments:
      sentiment_data = df[(df[aspect_column] == aspect) & (df[aspect_sentiment_column] == sentiment)][topic_column] # Filter topics for the current aspect and sentiment
      if not sentiment_data.empty:
        valid_sentiments.append(sentiment)
        term_counts = sentiment_data.explode().value_counts()
        valid_counts.append(term_counts)

    if not valid_sentiments:
      print(f"No terms found for aspect: {aspect}")
      continue

    fig, axs = plt.subplots(1, len(valid_sentiments), figsize=(6 * len(valid_sentiments), 6))
    fig.suptitle(f"Top terms for the Aspect: {aspect}")
    if len(valid_sentiments) == 1:
      axs = [axs]
    for i, (sentiment, counts) in enumerate(zip(valid_sentiments, valid_counts)):
      counts = counts[:min(15, len(counts))]
      axs[i].bar(counts.index, counts.values, color='skyblue')
      axs[i].set_title(f"{sentiment.capitalize()}")
      axs[i].set_ylabel('Frequency')
      axs[i].set_xlabel('Terms')
      axs[i].set_xticks(range(len(counts)))
      axs[i].set_xticklabels(counts.index, rotation=45, ha='right')
    plt.tight_layout()
    plt.show()


In [ ]:
def cross_product(list_1, list_2):
  return list(itertools.product(list_1, list_2))

def split_string_into_chunks(text, chunk_size=512):
  chunks = []
  for i in range(0, len(text), chunk_size):
    chunks.append(text[i:i + chunk_size])
  return chunks

def filter_and_summarize(df, aspect_column, text_column, aspects, sentiments, model_name="google/flan-t5-base"):
  total_comb = cross_product(aspects, sentiments)
  all_summary = {}
  for aspect, sentiment in total_comb:
    filtered_df = df[df[aspect_column].apply(lambda x: False if x.get(aspect) is None  else score_to_sent(x.get(aspect)['score']) == sentiment)]
    concatenated_text = "\n".join(filtered_df[text_column])
    if concatenated_text.strip() == "":
      all_summary[(aspect, sentiment)] = "No text is found for the aspect sentiment combination"
      continue
    summarizer = pipeline("summarization", model=model_name)
    temp_summary = ""
    for text in filtered_df[text_column]:
      chunk = text[0:min(len(text), 512)]
      if chunk.strip() == "":
        continue
      prompt = f"Summarize the following text with a focus on '{aspect}' ({sentiment} sentiment): {chunk}"
      temp_summary = temp_summary + summarizer(prompt, max_length=100, min_length=30, do_sample=False)[0]['summary_text']
      all_summary[(aspect, sentiment)] = temp_summary
  return all_summary


    # chunks = split_string_into_chunks(concatenated_text, chunk_size=512)
    # temp_summary = ""
    # for chunk in chunks:
    #   if chunk.strip() == "":
    #     continue

# Bangalore

In [ ]:
file_path = "/content/drive/MyDrive/Winter Project/data/DF Folder/absa_deberta_bangalore_20250107_16_56_34.json"
bangalore_df = pd.read_json(file_path, lines=True)
bangalore_df.head()

In [ ]:
bangalore_unique_topics = set().union(*bangalore_df['total_topics'])

In [ ]:
bangalore_topic_similarity, bangalore_unachieved_topics = get_similarity(topics = bangalore_unique_topics, categories = similarity_check_order, threshold=0.60)

In [ ]:
len(bangalore_topic_similarity)

In [ ]:
len(bangalore_unachieved_topics)

In [ ]:
bangalore_unachieved_topics

In [ ]:
bangalore_df["subcategory"] = bangalore_df.apply(lambda row: fix_aspects(sent_dict = row.deberta_aspect_sentiment, aspect_dict = bangalore_topic_similarity, sub_categ_list = sub_categ_reverse), axis = 1)

In [ ]:
bangalore_df["category"] = bangalore_df.apply(lambda row: merge_aspects(sent_dict = row.deberta_aspect_sentiment, aspect_dict = bangalore_topic_similarity, categ_list = categ_list), axis = 1)
bangalore_df["category"].value_counts()

In [ ]:
bangalore_df

In [ ]:
bangalore_df = overall_post_sentiment_using_aspect(bangalore_df)

In [ ]:
generate_wordcloud(bangalore_df)

In [ ]:
sentiment_distribution_and_top_terms(bangalore_df, order = sub_categ_reverse)

In [ ]:
raw_file_path = "/content/drive/MyDrive/Winter Project/data/New/raw_posts_bangalore.json"
bangalore_raw_df = pd.read_json(raw_file_path)
bangalore_comb_df = bangalore_df.join(bangalore_raw_df[["score", "upvote_ratio", "num_comments", "id"]].set_index("id"), on= ["id"], how = "left", rsuffix="_right")
bangalore_comb_df["deberta_aspect_sentiment"] = bangalore_comb_df["deberta_aspect_sentiment"].apply(delete_unnecessary_aspects)
bangalore_comb_df["total_votes"] = bangalore_comb_df.apply(lambda row: row.score * (1.0/(2*row.upvote_ratio-1)) if row.upvote_ratio!=0.5 else 0, axis = 1)
bangalore_comb_df["upvotes"] = bangalore_comb_df.apply(lambda row: ((row.score * (1.0/(2*row.upvote_ratio-1)))if row.upvote_ratio!=0.5 else 0)*row.upvote_ratio, axis = 1)
bangalore_comb_df["downvotes"] = bangalore_comb_df.apply(lambda row: ((row.score * (1.0/(2*row.upvote_ratio-1))) if row.upvote_ratio!=0.5 else 0)*(1-row.upvote_ratio), axis = 1)
bangalore_comb_df.head()

In [ ]:
bangalore_comb_df.loc[bangalore_comb_df['upvote_ratio']==0.5]

In [ ]:
bangalore_comb_df.describe()

In [ ]:
aspect_importance(bangalore_comb_df)
print("\n")
aspect_importance_score(bangalore_comb_df)
print("\n")
aspect_importance_upvotes(bangalore_comb_df)

In [ ]:
visualize_aspects(df=bangalore_comb_df, aspect_column="subcategory", text_column="text", aspects=sub_categ_reverse, top_n=15)

In [ ]:
topic_similarity_to_df(bangalore_topic_similarity)

In [ ]:
bangalore_df_flatten = transform_dataframe(df = bangalore_comb_df, aspect_column = "subcategory" , topics_column = "deberta_aspect_sentiment")
bangalore_top_terms = get_top_terms_per_aspect(dataframe = bangalore_df_flatten, sentiment = 'positive', aspect_column = 'aspects', term_column='topics', aspect_sentiment_column = 'aspect_sentiment', top_n=15)
# plot_barchart_per_aspect(aspect_terms = bangalore_top_terms, sentiment = 'positive')
required_sentiments = ["positive", "neutral","negative"]

In [ ]:

plot_barplots_by_aspect(df= bangalore_df_flatten, sentiments = required_sentiments, aspect_column = "aspects", topic_column = "topics", aspect_sentiment_column = "aspect_sentiment")

In [ ]:
plot_wordclouds_by_aspect(df= bangalore_df_flatten, sentiments = required_sentiments, aspect_column = "aspects", topic_column = "topics", aspect_sentiment_column = "aspect_sentiment")

In [ ]:
bangalore_summary = filter_and_summarize(bangalore_comb_df, aspect_column="subcategory", text_column="text", aspects=sub_categ_reverse, sentiments=required_sentiments)
bangalore_summary

# Delhi

In [ ]:
file_path = "/content/drive/MyDrive/Winter Project/data/DF Folder/absa_deberta_delhi_20250107_17_57_20.json"
delhi_df = pd.read_json(file_path, lines=True)
delhi_df.head()

In [ ]:
delhi_unique_topics = set().union(*delhi_df['total_topics'])

In [ ]:
delhi_topic_similarity, delhi_unachieved_topics = get_similarity(topics = delhi_unique_topics, categories = similarity_check_order, threshold=0.60)

In [ ]:
len(delhi_topic_similarity)

In [ ]:
len(delhi_unachieved_topics)

In [ ]:
delhi_unachieved_topics

In [ ]:
delhi_df["subcategory"] = delhi_df.apply(lambda row: fix_aspects(sent_dict = row.deberta_aspect_sentiment, aspect_dict = delhi_topic_similarity, sub_categ_list = sub_categ_reverse), axis = 1)

In [ ]:
delhi_df["category"] = delhi_df.apply(lambda row: merge_aspects(sent_dict = row.deberta_aspect_sentiment, aspect_dict = delhi_topic_similarity, categ_list = categ_list), axis = 1)
delhi_df["category"].value_counts()

In [ ]:
delhi_df

In [ ]:
delhi_df = overall_post_sentiment_using_aspect(delhi_df)

In [ ]:
generate_wordcloud(delhi_df)

In [ ]:
sentiment_distribution_and_top_terms(delhi_df, order = sub_categ_reverse)

In [ ]:
raw_file_path = "/content/drive/MyDrive/Winter Project/data/New/raw_posts_delhi.json"
delhi_raw_df = pd.read_json(raw_file_path)
delhi_comb_df = delhi_df.join(delhi_raw_df[["score", "upvote_ratio", "num_comments", "id"]].set_index("id"), on= ["id"], how = "left", rsuffix="_right")
delhi_comb_df["deberta_aspect_sentiment"] = delhi_comb_df["deberta_aspect_sentiment"].apply(delete_unnecessary_aspects)
delhi_comb_df["total_votes"] = delhi_comb_df.apply(lambda row: row.score * (1.0/(2*row.upvote_ratio-1)) if row.upvote_ratio!=0.5 else 0, axis = 1)
delhi_comb_df["upvotes"] = delhi_comb_df.apply(lambda row: ((row.score * (1.0/(2*row.upvote_ratio-1)))if row.upvote_ratio!=0.5 else 0)*row.upvote_ratio, axis = 1)
delhi_comb_df["downvotes"] = delhi_comb_df.apply(lambda row: ((row.score * (1.0/(2*row.upvote_ratio-1))) if row.upvote_ratio!=0.5 else 0)*(1-row.upvote_ratio), axis = 1)
delhi_comb_df.head()

In [ ]:
delhi_comb_df.loc[delhi_comb_df['upvote_ratio']==0.5]

In [ ]:
delhi_comb_df.describe()

In [ ]:
aspect_importance(delhi_comb_df)
print("\n")
aspect_importance_score(delhi_comb_df)
print("\n")
aspect_importance_upvotes(delhi_comb_df)

In [ ]:
visualize_aspects(df=delhi_comb_df, aspect_column="subcategory", text_column="text", aspects=sub_categ_reverse, top_n=15)

In [ ]:
topic_similarity_to_df(delhi_topic_similarity)

In [ ]:
delhi_df_flatten = transform_dataframe(df = delhi_comb_df, aspect_column = "subcategory" , topics_column = "deberta_aspect_sentiment")
delhi_top_terms = get_top_terms_per_aspect(dataframe = delhi_df_flatten, sentiment = 'positive', aspect_column = 'aspects', term_column='topics', aspect_sentiment_column = 'aspect_sentiment', top_n=15)
# plot_barchart_per_aspect(aspect_terms = delhi_top_terms, sentiment = 'positive')
required_sentiments = ["positive", "neutral","negative"]

In [ ]:

plot_barplots_by_aspect(df= delhi_df_flatten, sentiments = required_sentiments, aspect_column = "aspects", topic_column = "topics", aspect_sentiment_column = "aspect_sentiment")

In [ ]:
plot_wordclouds_by_aspect(df= delhi_df_flatten, sentiments = required_sentiments, aspect_column = "aspects", topic_column = "topics", aspect_sentiment_column = "aspect_sentiment")

In [ ]:

delhi_summary = filter_and_summarize(delhi_comb_df, aspect_column="subcategory", text_column="text", aspects=sub_categ_reverse, sentiments=["positive","negative"])
delhi_summary

# Mumbai

In [ ]:
file_path = "/content/drive/MyDrive/Winter Project/data/DF Folder/absa_deberta_mumbai_20250107_17_19_58.json"
mumbai_df = pd.read_json(file_path, lines=True)
mumbai_df.head()

In [ ]:
mumbai_unique_topics = set().union(*mumbai_df['total_topics'])

In [ ]:
mumbai_topic_similarity, mumbai_unachieved_topics = get_similarity(topics = mumbai_unique_topics, categories = similarity_check_order, threshold=0.60)

In [ ]:
len(mumbai_topic_similarity)

In [ ]:
len(mumbai_unachieved_topics)

In [ ]:
mumbai_unachieved_topics

In [ ]:
mumbai_df["subcategory"] = mumbai_df.apply(lambda row: fix_aspects(sent_dict = row.deberta_aspect_sentiment, aspect_dict = mumbai_topic_similarity, sub_categ_list = sub_categ_reverse), axis = 1)

In [ ]:
mumbai_df["category"] = mumbai_df.apply(lambda row: merge_aspects(sent_dict = row.deberta_aspect_sentiment, aspect_dict = mumbai_topic_similarity, categ_list = categ_list), axis = 1)
mumbai_df["category"].value_counts()

In [ ]:
mumbai_df

In [ ]:
mumbai_df = overall_post_sentiment_using_aspect(mumbai_df)

In [ ]:
generate_wordcloud(mumbai_df)

In [ ]:
sentiment_distribution_and_top_terms(mumbai_df, order = sub_categ_reverse)

In [ ]:
raw_file_path = "/content/drive/MyDrive/Winter Project/data/New/raw_posts_mumbai.json"
mumbai_raw_df = pd.read_json(raw_file_path)
mumbai_comb_df = mumbai_df.join(mumbai_raw_df[["score", "upvote_ratio", "num_comments", "id"]].set_index("id"), on= ["id"], how = "left", rsuffix="_right")
mumbai_comb_df["deberta_aspect_sentiment"] = mumbai_comb_df["deberta_aspect_sentiment"].apply(delete_unnecessary_aspects)
mumbai_comb_df["total_votes"] = mumbai_comb_df.apply(lambda row: row.score * (1.0/(2*row.upvote_ratio-1)) if row.upvote_ratio!=0.5 else 0, axis = 1)
mumbai_comb_df["upvotes"] = mumbai_comb_df.apply(lambda row: ((row.score * (1.0/(2*row.upvote_ratio-1)))if row.upvote_ratio!=0.5 else 0)*row.upvote_ratio, axis = 1)
mumbai_comb_df["downvotes"] = mumbai_comb_df.apply(lambda row: ((row.score * (1.0/(2*row.upvote_ratio-1))) if row.upvote_ratio!=0.5 else 0)*(1-row.upvote_ratio), axis = 1)
mumbai_comb_df.head()

In [ ]:
mumbai_comb_df.loc[mumbai_comb_df['upvote_ratio']==0.5]

In [ ]:
mumbai_comb_df.describe()

In [ ]:
aspect_importance(mumbai_comb_df)
print("\n")
aspect_importance_score(mumbai_comb_df)
print("\n")
aspect_importance_upvotes(mumbai_comb_df)

In [ ]:
visualize_aspects(df=mumbai_comb_df, aspect_column="subcategory", text_column="text", aspects=sub_categ_reverse, top_n=15)

In [ ]:
topic_similarity_to_df(mumbai_topic_similarity)

In [ ]:
mumbai_df_flatten = transform_dataframe(df = mumbai_comb_df, aspect_column = "subcategory" , topics_column = "deberta_aspect_sentiment")
mumbai_top_terms = get_top_terms_per_aspect(dataframe = mumbai_df_flatten, sentiment = 'positive', aspect_column = 'aspects', term_column='topics', aspect_sentiment_column = 'aspect_sentiment', top_n=15)
# plot_barchart_per_aspect(aspect_terms = mumbai_top_terms, sentiment = 'positive')
required_sentiments = ["positive", "neutral","negative"]

In [ ]:
plot_barplots_by_aspect(df= mumbai_df_flatten, sentiments = required_sentiments, aspect_column = "aspects", topic_column = "topics", aspect_sentiment_column = "aspect_sentiment")

In [ ]:
plot_wordclouds_by_aspect(df= mumbai_df_flatten, sentiments = required_sentiments, aspect_column = "aspects", topic_column = "topics", aspect_sentiment_column = "aspect_sentiment")

In [ ]:
mumbai_summary = filter_and_summarize(mumbai_comb_df, aspect_column="subcategory", text_column="text", aspects=sub_categ_reverse, sentiments=["positive","negative"])
mumbai_summary

# Visakhapatnam

In [ ]:
file_path = "/content/drive/MyDrive/Winter Project/data/DF Folder/absa_deberta_visakhapatnam_20250107_18_03_08.json"
vizag_df = pd.read_json(file_path, lines=True)
vizag_df.head()

In [ ]:
vizag_unique_topics = set().union(*vizag_df['total_topics'])

In [ ]:
vizag_topic_similarity, vizag_unachieved_topics = get_similarity(topics = vizag_unique_topics, categories = similarity_check_order, threshold=0.60)

In [ ]:
len(vizag_topic_similarity)

In [ ]:
len(vizag_unachieved_topics)

In [ ]:
vizag_unachieved_topics

In [ ]:
vizag_df["subcategory"] = vizag_df.apply(lambda row: fix_aspects(sent_dict = row.deberta_aspect_sentiment, aspect_dict = vizag_topic_similarity, sub_categ_list = sub_categ_reverse), axis = 1)

In [ ]:
vizag_df["category"] = vizag_df.apply(lambda row: merge_aspects(sent_dict = row.deberta_aspect_sentiment, aspect_dict = vizag_topic_similarity, categ_list = categ_list), axis = 1)
vizag_df["category"].value_counts()

In [ ]:
vizag_df

In [ ]:
vizag_df = overall_post_sentiment_using_aspect(vizag_df)

In [ ]:
generate_wordcloud(vizag_df)

In [ ]:
sentiment_distribution_and_top_terms(vizag_df, order = sub_categ_reverse)

In [ ]:
raw_file_path = "/content/drive/MyDrive/Winter Project/data/New/raw_posts_visakhapatnam.json"
vizag_raw_df = pd.read_json(raw_file_path)
vizag_comb_df = vizag_df.join(vizag_raw_df[["score", "upvote_ratio", "num_comments", "id"]].set_index("id"), on= ["id"], how = "left", rsuffix="_right")
vizag_comb_df["deberta_aspect_sentiment"] = vizag_comb_df["deberta_aspect_sentiment"].apply(delete_unnecessary_aspects)
vizag_comb_df["total_votes"] = vizag_comb_df.apply(lambda row: row.score * (1.0/(2*row.upvote_ratio-1)) if row.upvote_ratio!=0.5 else 0, axis = 1)
vizag_comb_df["upvotes"] = vizag_comb_df.apply(lambda row: ((row.score * (1.0/(2*row.upvote_ratio-1)))if row.upvote_ratio!=0.5 else 0)*row.upvote_ratio, axis = 1)
vizag_comb_df["downvotes"] = vizag_comb_df.apply(lambda row: ((row.score * (1.0/(2*row.upvote_ratio-1))) if row.upvote_ratio!=0.5 else 0)*(1-row.upvote_ratio), axis = 1)
vizag_comb_df.head()

In [ ]:
vizag_comb_df.loc[vizag_comb_df['upvote_ratio']==0.5]

In [ ]:
vizag_comb_df.describe()

In [ ]:
aspect_importance(vizag_comb_df)
print("\n")
aspect_importance_score(vizag_comb_df)
print("\n")
aspect_importance_upvotes(vizag_comb_df)

In [ ]:
visualize_aspects(df=vizag_comb_df, aspect_column="subcategory", text_column="text", aspects=sub_categ_reverse, top_n=15)

In [ ]:
topic_similarity_to_df(vizag_topic_similarity)

In [ ]:
vizag_df_flatten = transform_dataframe(df = vizag_comb_df, aspect_column = "subcategory" , topics_column = "deberta_aspect_sentiment")
vizag_top_terms = get_top_terms_per_aspect(dataframe = vizag_df_flatten, sentiment = 'positive', aspect_column = 'aspects', term_column='topics', aspect_sentiment_column = 'aspect_sentiment', top_n=15)
# plot_barchart_per_aspect(aspect_terms = vizag_top_terms, sentiment = 'positive')
required_sentiments = ["positive", "neutral","negative"]

In [ ]:
plot_barplots_by_aspect(df= vizag_df_flatten, sentiments = required_sentiments, aspect_column = "aspects", topic_column = "topics", aspect_sentiment_column = "aspect_sentiment")

In [ ]:
plot_wordclouds_by_aspect(df= vizag_df_flatten, sentiments = required_sentiments, aspect_column = "aspects", topic_column = "topics", aspect_sentiment_column = "aspect_sentiment")

In [ ]:
vizag_summary = filter_and_summarize(vizag_comb_df, aspect_column="subcategory", text_column="text", aspects=sub_categ_reverse, sentiments=["positive","negative"])
vizag_summary

# New York

In [ ]:
file_path = "/content/drive/MyDrive/Winter Project/data/DF Folder/absa_deberta_newyork_20250107_18_07_30.json"
newyork_df = pd.read_json(file_path, lines=True)
newyork_df.head()

In [ ]:
newyork_unique_topics = set().union(*newyork_df['total_topics'])

In [ ]:
newyork_topic_similarity, newyork_unachieved_topics = get_similarity(topics = newyork_unique_topics, categories = similarity_check_order, threshold=0.60)

In [ ]:
len(newyork_topic_similarity)

In [ ]:
len(newyork_unachieved_topics)

In [ ]:
newyork_unachieved_topics

In [ ]:
newyork_df["subcategory"] = newyork_df.apply(lambda row: fix_aspects(sent_dict = row.deberta_aspect_sentiment, aspect_dict = newyork_topic_similarity, sub_categ_list = sub_categ_reverse), axis = 1)

In [ ]:
newyork_df["category"] = newyork_df.apply(lambda row: merge_aspects(sent_dict = row.deberta_aspect_sentiment, aspect_dict = newyork_topic_similarity, categ_list = categ_list), axis = 1)
newyork_df["category"].value_counts()

In [ ]:
newyork_df

In [ ]:
newyork_df = overall_post_sentiment_using_aspect(newyork_df)

In [ ]:
generate_wordcloud(newyork_df)

In [ ]:
sentiment_distribution_and_top_terms(newyork_df, order = sub_categ_reverse)

In [ ]:
raw_file_path = "/content/drive/MyDrive/Winter Project/data/New/raw_posts_newyorkcity.json"
newyork_raw_df = pd.read_json(raw_file_path)
newyork_comb_df = newyork_df.join(newyork_raw_df[["score", "upvote_ratio", "num_comments", "id"]].set_index("id"), on= ["id"], how = "left", rsuffix="_right")
newyork_comb_df["deberta_aspect_sentiment"] = newyork_comb_df["deberta_aspect_sentiment"].apply(delete_unnecessary_aspects)
newyork_comb_df["total_votes"] = newyork_comb_df.apply(lambda row: row.score * (1.0/(2*row.upvote_ratio-1)) if row.upvote_ratio!=0.5 else 0, axis = 1)
newyork_comb_df["upvotes"] = newyork_comb_df.apply(lambda row: ((row.score * (1.0/(2*row.upvote_ratio-1)))if row.upvote_ratio!=0.5 else 0)*row.upvote_ratio, axis = 1)
newyork_comb_df["downvotes"] = newyork_comb_df.apply(lambda row: ((row.score * (1.0/(2*row.upvote_ratio-1))) if row.upvote_ratio!=0.5 else 0)*(1-row.upvote_ratio), axis = 1)
newyork_comb_df.head()

In [ ]:
newyork_comb_df.loc[newyork_comb_df['upvote_ratio']==0.5]

In [ ]:
newyork_comb_df.describe()

In [ ]:
aspect_importance(newyork_comb_df)
print("\n")
aspect_importance_score(newyork_comb_df)
print("\n")
aspect_importance_upvotes(newyork_comb_df)

In [ ]:
visualize_aspects(df=newyork_comb_df, aspect_column="subcategory", text_column="text", aspects=sub_categ_reverse, top_n=15)

In [ ]:
topic_similarity_to_df(newyork_topic_similarity)

In [ ]:
newyork_df_flatten = transform_dataframe(df = newyork_comb_df, aspect_column = "subcategory" , topics_column = "deberta_aspect_sentiment")
newyork_top_terms = get_top_terms_per_aspect(dataframe = newyork_df_flatten, sentiment = 'positive', aspect_column = 'aspects', term_column='topics', aspect_sentiment_column = 'aspect_sentiment', top_n=15)
# plot_barchart_per_aspect(aspect_terms = newyork_top_terms, sentiment = 'positive')
required_sentiments = ["positive", "neutral","negative"]

In [ ]:
plot_barplots_by_aspect(df= newyork_df_flatten, sentiments = required_sentiments, aspect_column = "aspects", topic_column = "topics", aspect_sentiment_column = "aspect_sentiment")

In [ ]:
plot_wordclouds_by_aspect(df= newyork_df_flatten, sentiments = required_sentiments, aspect_column = "aspects", topic_column = "topics", aspect_sentiment_column = "aspect_sentiment")

In [ ]:
newyork_summary = filter_and_summarize(newyork_comb_df, aspect_column="subcategory", text_column="text", aspects=sub_categ_reverse, sentiments=["positive","negative"])
newyork_summary